# RBY1 Grasp Execution - Interactive Notebook (Sim + Real)

This notebook allows you to execute the grasp pipeline step by step, testing first on the simulator before executing on the real robot.

## 1. Imports and Setup

In [1]:
import os
import sys
import numpy as np
import torch
from PIL import Image
from scipy.spatial.transform import Rotation as R

import rby1_sdk as rby

import ok_robot_manipulation
from ok_robot_manipulation.src.grasp_predict import GraspPredictor
from ok_robot_manipulation.src.utils.rby1 import (
    connect_to_robot,
    move_to_zero_position,
    move_to_ready_position,
    get_camera_pose_from_robot,
    execute_grasp_motion,
    print_pose_error
)
from ok_robot_manipulation.src.utils.camera_client import CameraClient

print("✓ Imports successful")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
checking license on /home/usd/Research/ok_robot_real/ok_robot_manipulation/src/gsnet.so
[2026-01-16 16:26:55.934] [info] [FlexivLic] public key JebeomChae.public_key & signature JebeomChae.signature are matched
license passed: True, state: FvrLicenseState.PASSED
[2026-01-16 16:26:55.934] [info] [FlexivLic] license /home/usd/Research/ok_robot_real/ok_robot_manipulation/src/license/JebeomChae.lic check passed.


/home/usd/anaconda3/envs/ok-real/lib/python3.10/site-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(
/home/usd/anaconda3/envs/ok-real/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/usd/anaconda3/envs/ok-real/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead

✓ Imports successful


/home/usd/anaconda3/envs/ok-real/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/usd/anaconda3/envs/ok-real/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## 2. Configuration

In [2]:
# Configuration parameters
DATA_DIR = "../ok_robot_manipulation/src/example_data_rby1/"
SIMULATOR_ADDRESS = "0.0.0.0:50051"  # Simulator robot address
REAL_ROBOT_ADDRESS = "192.168.12.1:50051"  # Real robot address
HEAD_CAMERA_LINK = "link_head_2"
PRE_GRASP_OFFSET = 0.07  # meters
DEBUG = True  # Enable visualization

# Camera server configuration
CAMERA_SERVER_HOST = "192.168.0.39"  # Camera server IP
CAMERA_SERVER_PORT = 5556

# Grasp predictor configuration
PREDICTOR_CONFIG = {
    'environment': DATA_DIR,
    'debug': DEBUG,
    'predictor_headless': False,
    'checkpoint_path': '/home/usd/Research/ok_robot_real/ok_robot_manipulation/src/checkpoints/checkpoint_detection.tar',
    'query': 'scissors', # 'bandage', 'bottle'
    'open_communication': False,
}

print(f"Data directory: {DATA_DIR}")
print(f"Simulator address: {SIMULATOR_ADDRESS}")
print(f"Real robot address: {REAL_ROBOT_ADDRESS}")
print(f"Camera server: {CAMERA_SERVER_HOST}:{CAMERA_SERVER_PORT}")
print(f"Debug mode: {DEBUG}")

Data directory: ../ok_robot_manipulation/src/example_data_rby1/
Simulator address: 0.0.0.0:50051
Real robot address: 192.168.12.1:50051
Camera server: 192.168.0.39:5556
Debug mode: True


## 3. Capture RGB and Depth from Camera Server

In [ ]:
# Connect to camera server and capture images
print(f"Connecting to camera server at {CAMERA_SERVER_HOST}:{CAMERA_SERVER_PORT}...")
camera_client = CameraClient(CAMERA_SERVER_HOST, CAMERA_SERVER_PORT)

if not camera_client.connect(timeout=5.0):
    raise ConnectionError(f"Failed to connect to camera server at {CAMERA_SERVER_HOST}:{CAMERA_SERVER_PORT}")

print("✓ Connected to camera server")

# Check server status
status = camera_client.get_status()
if status:
    print(f"Camera server status: {status}")

# Capture RGB-D images
print("Capturing RGB-D images...")
data = camera_client.capture()

if data is None:
    raise RuntimeError("Failed to capture images from camera server")

# Extract images
rgb_array = data['rgb']  # numpy array (H, W, 3)
depth_array = data['depth']  # numpy array (H, W) in mm

# Convert RGB numpy array to PIL Image for predictor
color_image = Image.fromarray(rgb_array)

# Convert depth from mm to meters for predictor
depths = depth_array.astype(np.float32) / 1000.0

print(f"✓ Captured RGB image: {color_image.size}")
print(f"✓ Captured depth data: {depths.shape}")
print(f"  Depth range: {depths[depths > 0].min():.3f}m - {depths.max():.3f}m")

# Save captured images to DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)

rgb_path = os.path.join(DATA_DIR, "rgb.png")
depth_path = os.path.join(DATA_DIR, "head_camera_depth.npy")

color_image.save(rgb_path)
np.save(depth_path, depths)

print(f"✓ Saved RGB image to: {rgb_path}")
print(f"✓ Saved depth data to: {depth_path}")

# Display RGB image
from IPython.display import display
display(color_image)

## 4. Connect to Simulator Robot

In [3]:
# Connect to simulator robot
print("Connecting to SIMULATOR robot...")
sim_robot, sim_model = connect_to_robot(SIMULATOR_ADDRESS)

print("\n✓ Simulator robot connected and initialized")

Connecting to SIMULATOR robot...
[INFO] Connecting to robot at 0.0.0.0:50051...
[INFO] Connected to robot successfully
[INFO] Initializing robot control system...
[INFO] Turning on power...
[INFO] Enabling servo motors...
[INFO] Resetting fault control manager...
[INFO] Enabling control manager...
[INFO] Robot control system initialized successfully

✓ Simulator robot connected and initialized


## 5. Connect to Real Robot

In [4]:
# Connect to real robot
print("Connecting to REAL robot...")
real_robot, real_model = connect_to_robot(REAL_ROBOT_ADDRESS)

print("\n✓ Real robot connected and initialized")

Connecting to REAL robot...
[INFO] Connecting to robot at 192.168.12.1:50051...
[INFO] Connected to robot successfully
[INFO] Initializing robot control system...
[INFO] Turning on power...
[INFO] Enabling servo motors...
[INFO] Resetting fault control manager...
[INFO] Enabling control manager...
[INFO] Robot control system initialized successfully

✓ Real robot connected and initialized


## 6. Move Simulator to Zero Position

In [5]:
# Move simulator robot to home position
print("Moving SIMULATOR to zero position...")
success = move_to_zero_position(sim_robot, minimum_time=5.0)

if success:
    print("\n✓ Simulator moved to zero position")
else:
    print("\n✗ Simulator failed to move to zero position")

Moving SIMULATOR to zero position...
[INFO] Moving body to zero position...
[INFO] Body moved to zero position
[INFO] Moving head to zero position...
[INFO] Head moved to zero position

✓ Simulator moved to zero position


## 7. Move Real Robot to Zero Position

In [ ]:
# Move real robot to home position
print("Moving REAL ROBOT to zero position...")
success = move_to_zero_position(real_robot, minimum_time=5.0)

if success:
    print("\n✓ Real robot moved to zero position")
else:
    print("\n✗ Real robot failed to move to zero position")

## 8. Move Simulator to Ready Position

In [ ]:
# Move simulator robot to manipulation-ready pose
print("Moving SIMULATOR to ready position...")
success = move_to_ready_position(sim_robot, minimum_time=5.0)

if success:
    print("\n✓ Simulator moved to ready position")
else:
    print("\n✗ Simulator failed to move to ready position")

## 9. Move Real Robot to Ready Position

In [ ]:
# Move real robot to manipulation-ready pose
print("Moving REAL ROBOT to ready position...")
success = move_to_ready_position(real_robot, minimum_time=5.0)

if success:
    print("\n✓ Real robot moved to ready position")
else:
    print("\n✗ Real robot failed to move to ready position")

## 11. Get Camera Pose from Real Robot

In [ ]:
# Compute camera pose from real robot with proper offset and rotation transformation
camera_pose_in_world = get_camera_pose_from_robot(real_robot, real_model, HEAD_CAMERA_LINK)

print("\n✓ Camera pose computed from REAL ROBOT")
print(f"  Position: {camera_pose_in_world[0]}")
print(f"  Quaternion (w,x,y,z): {camera_pose_in_world[1]}")

## 12. Initialize Grasp Predictor

In [ ]:
# Initialize Grasp Predictor using add_predictor_args defaults, then overlay PREDICTOR_CONFIG
import argparse
from ok_robot_manipulation.src.grasp_predict import GraspPredictor, add_predictor_args

# Build default args from library
parser = argparse.ArgumentParser()
add_predictor_args(parser)
args = parser.parse_args(args=[])  # get defaults without CLI

# Overlay notebook config dict onto args
for k, v in PREDICTOR_CONFIG.items():
    setattr(args, k, v)

# Backward compatibility: set headless from predictor_headless
args.headless = getattr(args, 'predictor_headless', False)

print("Initializing Grasp Predictor...")
print(f"  Environment: {args.environment}")
print(f"  Query: {getattr(args, 'query', None)}")
print(f"  Headless: {args.headless}")

grasp_predictor = GraspPredictor(args)
print("✓ Grasp predictor initialized")

## 13. Run Grasp Prediction

In [ ]:
# Capture RGB-D images
print("Capturing RGB-D images...")
data = camera_client.capture()

if data is None:
    raise RuntimeError("Failed to capture images from camera server")

# Extract images
rgb_array = data['rgb']  # numpy array (H, W, 3)
depth_array = data['depth']  # numpy array (H, W) in mm

# Convert RGB numpy array to PIL Image for predictor
color_image = Image.fromarray(rgb_array)

# Convert depth from mm to meters for predictor
depths = depth_array.astype(np.float32) / 1000.0

# Display RGB image
from IPython.display import display
display(color_image)

# Run grasp prediction with AnyGrasp
print("Running grasp prediction...")
final_grasp_pose, model_transition = grasp_predictor.predict(
    rgb_image=color_image,
    depths=depths,
    head_link_pose_in_world=camera_pose_in_world,
)

if final_grasp_pose is not None:
    pos_np = final_grasp_pose[0:3]
    quat_wxyz_np = final_grasp_pose[3:7]

    print("\n" + "="*50)
    print(" >> SUCCESS: Grasp Target Found")
    print("="*50)
    
    # Select arm based on x position
    if model_transition[0] < 0:
        arm_str = "LEFT Arm"
        arm_side = "left"
    else:
        arm_str = "RIGHT Arm"
        arm_side = "right"
        
    print(f" [Selected Arm] : {arm_str}")
    print(f" [Target Pos]   : {pos_np}")
    print(f" [Target Quat]  : {quat_wxyz_np} (w, x, y, z)")
    print("="*50 + "\n")
else:
    print("\n" + "="*50)
    print(" >> FAIL: No Grasp Target Found")
    print("="*50 + "\n")
    pos_np = None
    quat_wxyz_np = None
    arm_side = None

## 14. Execute Grasp Motion on SIMULATOR

**Test the grasp on the simulator first to verify it's safe and correct.**

In [ ]:
# Execute grasp on SIMULATOR if target was found
# ← Modify impedance gains directly below to tune grasp behavior
if final_grasp_pose is not None:
    print("=" * 60)
    print("EXECUTING GRASP ON SIMULATOR")
    print("=" * 60)
    print(f"Using {arm_side} arm")
    print(f"Pre-grasp offset: {PRE_GRASP_OFFSET}m")
    
    sim_success = execute_grasp_motion(
        sim_robot, sim_model,
        pos_np, quat_wxyz_np,
        arm_side,
        pre_grasp_offset=PRE_GRASP_OFFSET,
    )
    
    if sim_success:
        print("\n" + "="*60)
        print(" >> SIMULATOR: Grasp Execution Completed Successfully!")
        print("="*60 + "\n")
        print("✓ Ready to execute on real robot in next cell")
    else:
        print("\n" + "="*60)
        print(" >> SIMULATOR: Grasp Execution Failed")
        print("="*60 + "\n")
        print("⚠️ Fix issues before running on real robot!")
else:
    print("Cannot execute grasp - no target found in prediction step")

## 15. Execute Grasp Motion on REAL ROBOT

**⚠️ WARNING:** This will move the REAL ROBOT! 
- Ensure simulator execution was successful first
- Make sure the workspace is clear
- Be ready to emergency stop if needed

In [ ]:
# Execute grasp on REAL ROBOT if target was found and simulator succeeded
if final_grasp_pose is not None:
    print("=" * 60)
    print("EXECUTING GRASP ON REAL ROBOT")
    print("=" * 60)
    print(f"Using {arm_side} arm")
    print(f"Pre-grasp offset: {PRE_GRASP_OFFSET}m")
    
    # Gripper parameters (remote control only)
    gripper_close_position = 0.5  # Normalized 0.0-1.0 (0.0 = fully closed, 1.0 = fully open)
    gripper_hold_time = 1.0       # Seconds
    remote_gripper_host = "192.168.0.39"  # ← set to the gripper PC IP; None to disable
    remote_gripper_port = 5009
    
    print(f"Gripper close position: {gripper_close_position}")
    print(f"Gripper hold time: {gripper_hold_time}s")
    print(f"Remote gripper: {remote_gripper_host}:{remote_gripper_port}\n")
    
    real_success = execute_grasp_motion(
        real_robot, real_model,
        pos_np, quat_wxyz_np,
        arm_side,
        pre_grasp_offset=PRE_GRASP_OFFSET,
        gripper_close_position=gripper_close_position,
        gripper_hold_time=gripper_hold_time,
        remote_gripper_host=remote_gripper_host,
        remote_gripper_port=remote_gripper_port,
    )
    
    if real_success:
        print("\n" + "="*60)
        print(" >> REAL ROBOT: Grasp Execution Completed Successfully!")
        print("="*60 + "\n")
    else:
        print("\n" + "="*60)
        print(" >> REAL ROBOT: Grasp Execution Failed")
        print("="*60 + "\n")
else:
    print("Cannot execute grasp - no target found in prediction step")

## 16. Optional: Manual Pose Error Checking

You can manually check the current pose error for any end-effector link on either robot:

In [ ]:
# Example: Check current pose for both robots
if final_grasp_pose is not None:
    # Create target transformation matrix
    rot = R.from_quat([quat_wxyz_np[1], quat_wxyz_np[2], quat_wxyz_np[3], quat_wxyz_np[0]])  # [x,y,z,w]
    target_rot_matrix = rot.as_matrix()
    
    T_target = np.eye(4)
    T_target[0:3, 0:3] = target_rot_matrix
    T_target[0:3, 3] = pos_np
    
    ee_link = f"ee_{arm_side}"
    
    print("SIMULATOR:")
    print_pose_error(sim_robot, sim_model, ee_link, T_target, "Simulator Current State")
    
    print("\nREAL ROBOT:")
    print_pose_error(real_robot, real_model, ee_link, T_target, "Real Robot Current State")
else:
    print("No target pose available")

## Notes

- **Dual Robot Setup**: Commands are sent to simulator first, then to real robot in separate cells
- **Safety**: Always verify simulator execution before running on real hardware
- Each cell can be run independently after the previous cells have been executed
- You can re-run the prediction step without reconnecting to the robots
- The grasp execution uses impedance control with:
  - Translation weight: 1500 N/m
  - Rotation weight: 100 Nm/rad
  - Damping ratio: 1.0 (critically damped)
- Camera offset from head_link_2: +0.05m in Z-axis with rotation transformation
- Camera pose is obtained from the real robot for accurate prediction

## 17. Disconnect Camera Server

In [ ]:
camera_client.disconnect()
print("✓ Disconnected from camera server")